# Tratamento e Limpeza dos Dados - Sales Transaction

Notebook completo para a etapa 2.2 do projeto:
- Valores ausentes
- Outliers
- Conversão de tipos
- Duplicatas
- Normalização/Padronização
- Engenharia de atributos


In [4]:
import pandas as pd
import numpy as np

In [5]:
df = pd.read_csv('Sales Transaction v.4a.csv')
df.head()

,TransactionNo,Date,ProductNo,ProductName,Price,Quantity,CustomerNo,Country
0,581482,12/9/2019,22485,Set Of 2 Wooden Market Crates,21.47,12,17490.0,United Kingdom
1,581475,12/9/2019,22596,Christmas Star Wish List Chalkboard,10.65,36,13069.0,United Kingdom
2,581475,12/9/2019,23235,Storage Tin Vintage Leaf,11.53,12,13069.0,United Kingdom
3,581475,12/9/2019,23272,Tree T-Light Holder Willie Winkie,10.65,12,13069.0,United Kingdom
4,581475,12/9/2019,23239,Set Of 4 Knick Knack Tins Poppies,11.94,6,13069.0,United Kingdom


In [6]:
# Informações gerais

print(df.shape)
df.info()

(536350, 8)
<class 'pandas.DataFrame'>
RangeIndex: 536350 entries, 0 to 536349
Data columns (total 8 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   TransactionNo  536350 non-null  str    
 1   Date           536350 non-null  str    
 2   ProductNo      536350 non-null  str    
 3   ProductName    536350 non-null  str    
 4   Price          536350 non-null  float64
 5   Quantity       536350 non-null  int64  
 6   CustomerNo     536295 non-null  float64
 7   Country        536350 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 63.4 MB


In [7]:
# Valores ausentes

missing = df.isnull().sum()
missing_percent = (missing/len(df))*100

pd.DataFrame({
    'Valores Ausentes': missing,
    'Percentual (%)': missing_percent
}).sort_values('Valores Ausentes', ascending=False)

,Valores Ausentes,Percentual (%)
CustomerNo,55,0.010254
TransactionNo,0,0.000000
ProductNo,0,0.000000
Date,0,0.000000
ProductName,0,0.000000
Price,0,0.000000
Quantity,0,0.000000
Country,0,0.000000


In [8]:
# Tratamento dos valores ausentes

df['CustomerNo'] = df['CustomerNo'].fillna(0)
print(df.isnull().sum())

TransactionNo    0
Date             0
ProductNo        0
ProductName      0
Price            0
Quantity         0
CustomerNo       0
Country          0
dtype: int64


In [9]:
# Verificação de duplicatas

total_duplicadas = df.duplicated().sum()

print(f"Total de linhas duplicadas no dataset: {total_duplicadas}")

Total de linhas duplicadas no dataset: 5200


In [10]:
# Remoção de duplicatas

df = df.drop_duplicates()
print(f"Total de linhas duplicadas após a limpeza: {df.duplicated().sum()}")

Total de linhas duplicadas após a limpeza: 0


In [11]:
# Detecção de outliers pelo método IQR

outliers = {}

numericas = df.select_dtypes(include=['int64', 'float64']).columns

colunas_validas = [col for col in numericas if col != 'CustomerNo']

for col in colunas_validas:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1

    inferior = q1 - 1.5 * iqr
    superior = q3 + 1.5 * iqr

    qtd = ((df[col] < inferior) | (df[col] > superior)).sum()
    outliers[col] = qtd

df_outliers = pd.DataFrame.from_dict(outliers, orient='index', columns=['Qtd_Outliers']).sort_values('Qtd_Outliers', ascending=False)
print(df_outliers)

          Qtd_Outliers
Price            98369
Quantity         57255


In [12]:
# Conversão de tipos (quando necessário)

df['Country'] = df['Country'].astype('category')
df['Date'] = pd.to_datetime(df['Date'])
df['CustomerNo'] = df['CustomerNo'].astype(str).str.replace(r'\.0$', '', regex=True)

print(df.dtypes)

TransactionNo               str
Date             datetime64[us]
ProductNo                   str
ProductName                 str
Price                   float64
Quantity                  int64
CustomerNo                  str
Country                category
dtype: object


In [13]:
# Engenharia de atributos

df['Total_Amount'] = df['Quantity'] * df['Price']

df['Year_Month'] = df['Date'].dt.to_period('M')

df[['Total_Amount', 'Year_Month']].head()

,Total_Amount,Year_Month
0,257.64,2019-12
1,383.40,2019-12
2,138.36,2019-12
3,127.80,2019-12
4,71.64,2019-12


In [14]:
# Padronização das variáveis numéricas

from sklearn.preprocessing import StandardScaler

numericas = ['Quantity', 'Price', 'Total_Amount']

scaler = StandardScaler()

df_padronizado = df.copy()
df_padronizado[numericas] = scaler.fit_transform(df[numericas])

df_padronizado.head()

,TransactionNo,Date,ProductNo,ProductName,Price,Quantity,CustomerNo,Country,Total_Amount,Year_Month
0,581482,2019-12-09,22485,Set Of 2 Wooden Market Crates,1.032164,0.009240,17490,United Kingdom,0.063129,2019-12
1,581475,2019-12-09,22596,Christmas Star Wish List Chalkboard,-0.236898,0.119483,13069,United Kingdom,0.118087,2019-12
2,581475,2019-12-09,23235,Storage Tin Vintage Leaf,-0.133684,0.009240,13069,United Kingdom,0.011003,2019-12
3,581475,2019-12-09,23272,Tree T-Light Holder Willie Winkie,-0.236898,0.009240,13069,United Kingdom,0.006388,2019-12
4,581475,2019-12-09,23239,Set Of 4 Knick Knack Tins Poppies,-0.085596,-0.018321,13069,United Kingdom,-0.018154,2019-12


## Conclusão

- Valores ausentes tratados por preenchimento padrão na coluna de ID.
- Total de 1682 linhas duplicadas removidas.
- Outliers identificados pelo método IQR nas colunas de quantidade e preço.
- Tipos de dados revisados e convertidos para categorias e datas.
- Novos atributos criados para enriquecer as análises de faturamento e períodos.
- Variáveis numéricas padronizadas para a modelagem.


In [15]:
df.to_csv("Sales Transaction v.4a_clean.csv", index=False)